# 02. Метео-классификатор условий: дождь, снег, туман, град/гроза (угрозы №7, 8, 9, 5)

## Задача
По полям, которые есть в ТЗ, определить тип погоды:
| Поле | Поток ТЗ | В данных METAR |
|---|---|---|
| `temperature`, `humidity` | БВС-5 | `tmpc`, `relh` (+ `dwpc`) |
| `wind_speed`, `wind_gust` | БВС-5 | `wind_ms`, `gust_ms` |
| `visibility_m` | НСУ-3 (внешний прогноз/наблюдение) | `vis_m` |

Классы: `dry, rain, snow, fog, aerosol, hail_ts` (приоритет при совпадении: hail_ts > snow > rain > fog > aerosol > dry). Метки берутся из кодов present weather METAR (RA, SN, FG, GR, TS...), то есть **настоящие, не синтетические**.

## Почему это полезно без видео
Это единственный блок проекта, где есть реальная разметка. Результаты задают «метео-prior» для остальных ноутбуков (интеграция, понижение доверия к детекциям).

> Важно: метки привязаны к аэродромной станции, а не к дрону. Модель показывает, что *сочетание T, RH, ветра* информативно для типа погоды, а не точность на борту.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

from src.data_metar import load_or_fallback, CLASSES
from src.metrics import ece, reliability_curve
from src.detectors import hysteresis
plt.rcParams["figure.dpi"] = 110
RNG = 0

## 1. Данные
Загрузка из Iowa Environmental Mesonet (METAR + SPECI), кэш в `data/`. Если сети нет, `load_or_fallback` **выдаёт предупреждение** и подставляет синтетический ряд. В этом случае метки порождены правилами по T/RH, и результаты ниже круговые: годятся лишь как проверка пайплайна.

In [ ]:
STATIONS = ["EDDF", "EDDM", "EDDH", "EDDB"]      # Франкфурт, Мюнхен, Гамбург, Берлин
YEARS = list(range(2017, 2024))
data, SOURCE = load_or_fallback(STATIONS, YEARS, cache_dir=str(ROOT / "data"))
print("Источник данных:", SOURCE, "| строк:", len(data))
if SOURCE != "IEM":
    print("!!! РЕЗУЛЬТАТЫ НИЖЕ НЕ ОТНОСЯТСЯ К РЕАЛЬНОЙ ПОГОДЕ !!!")

FEATS_RAW = ["tmpc", "relh", "wind_ms"]
FEATS_ONBOARD = ["tmpc", "relh", "Td", "Tw", "spread", "wind_ms", "gust_ms", "dT_1h", "dT_3h", "dspread_1h"]
FEATS_VIS = FEATS_ONBOARD + ["vis_m"]

df = data.dropna(subset=FEATS_ONBOARD + ["label"]).copy()
df["year"] = df.index.year
print(df.label.value_counts().reindex(CLASSES).fillna(0).astype(int).to_string())

In [ ]:
ct = df.groupby(["station", "label"]).size().unstack(fill_value=0).reindex(columns=CLASSES, fill_value=0)
ax = (ct.div(ct.sum(1), axis=0) * 100).plot.bar(stacked=True, figsize=(7, 3.2))
ax.set_ylabel("% часов"); ax.set_title("Доля классов по станциям"); plt.legend(ncol=3, fontsize=8); plt.show()

## 2. Признаки и разбиение
- Td и Tw (Magnus, Stull) считаются из T и RH. **Tw** разделяет дождь и снег лучше, чем одна T.
- Тренды `dT_1h`, `dT_3h`, `dspread_1h`: в полёте их можно считать по скользящему окну БВС-5.
- **Три набора признаков:** `raw` (T, RH, ветер), `onboard` (производные + тренды), `+vis` (добавлена видимость из НСУ-3). Видимость сильно «подсказывает» туман (метка FG определяется самой видимостью), поэтому её вклад показываем отдельно.
- **Разбиение по времени:** обучение на всех годах кроме последнего, тест на последнем. Случайное перемешивание часов дало бы утечку из-за автокорреляции.

In [ ]:
test_year = YEARS[-1]
tr, te = df[df.year < test_year], df[df.year == test_year]
print(f"train {len(tr)}, test {len(te)} (год {test_year})")

def fit_eval(feats, model="hgb", balanced=True, train=tr, test=te):
    if model == "logreg":
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, class_weight="balanced" if balanced else None))
    else:
        m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=RNG,
                                           class_weight="balanced" if balanced else None)
    m.fit(train[feats], train.label)
    pred = m.predict(test[feats])
    labels = sorted(set(test.label) | set(pred))
    return m, pred, f1_score(test.label, pred, average="macro", labels=sorted(set(test.label)), zero_division=0)

def rule_baseline(d):
    """Наивные пороги без обучения: «туман» по spread и ветру, «осадки» по RH, тип осадков по Tw."""
    p = np.full(len(d), "dry", dtype=object)
    wet = d.relh.values >= 90
    p[wet & (d.Tw.values < 1.0)] = "snow"
    p[wet & (d.Tw.values >= 1.0)] = "rain"
    p[(d.spread.values < 0.8) & (d.wind_ms.values < 3)] = "fog"
    return p

## 3. Сравнение моделей

In [ ]:
rows = []
p = rule_baseline(te)
rows.append(("rule (пороги Tw/RH)", "onboard", f1_score(te.label, p, average="macro", labels=sorted(set(te.label)), zero_division=0)))
for name, feats in [("raw", FEATS_RAW), ("onboard", FEATS_ONBOARD), ("+vis", FEATS_VIS)]:
    for model in ("logreg", "hgb"):
        _, _, f1 = fit_eval(feats, model)
        rows.append((model, name, f1))
res_tab = pd.DataFrame(rows, columns=["модель", "признаки", "macro-F1"]).round(3)
display(res_tab.pivot(index="модель", columns="признаки", values="macro-F1"))

In [ ]:
m, pred, _ = fit_eval(FEATS_ONBOARD, "hgb")
labs = [c for c in CLASSES if c in set(te.label) | set(pred)]
print(classification_report(te.label, pred, labels=labs, zero_division=0))
cm = confusion_matrix(te.label, pred, labels=labs, normalize="true")
fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(labs)), labs, rotation=45); ax.set_yticks(range(len(labs)), labs)
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="k" if v < .6 else "w", fontsize=8)
ax.set_xlabel("предсказано"); ax.set_ylabel("истина"); ax.set_title("HGB, onboard-признаки"); plt.show()

## 4. Абляция по станциям (leave-one-station-out)
Обучаемся на трёх станциях, тестируем на четвёртой: проверка переносимости на другой климат/оборудование.

In [ ]:
rows = []
for st in STATIONS:
    a, b = df[df.station != st], df[df.station == st]
    _, _, f1 = fit_eval(FEATS_ONBOARD, "hgb", train=a, test=b)
    rows.append((st, round(f1, 3)))
display(pd.DataFrame(rows, columns=["тест-станция", "macro-F1"]).set_index("тест-станция").T)

## 5. Вклад признаков
Permutation importance на подвыборке теста (падение macro-F1 при перемешивании признака).

In [ ]:
sub = te.sample(min(6000, len(te)), random_state=RNG)
pi = permutation_importance(m, sub[FEATS_ONBOARD], sub.label, n_repeats=5, random_state=RNG, scoring="f1_macro")
imp = pd.Series(pi.importances_mean, index=FEATS_ONBOARD).sort_values()
imp.plot.barh(figsize=(5, 3.2)); plt.xlabel("падение macro-F1"); plt.title("Важность признаков"); plt.tight_layout(); plt.show()

## 6. Калибровка вероятностей
Выход пойдёт в Confidence Monitoring, поэтому вероятности должны быть честными. Модель с `class_weight="balanced"` хорошо находит редкие классы, но **завышает** их вероятность. Сравним с моделью без весов для события «осадки» (rain ∪ snow).

In [ ]:
def p_precip(model, d, feats):
    P = model.predict_proba(d[feats]); cl = list(model.classes_)
    return sum(P[:, cl.index(c)] for c in ("rain", "snow") if c in cl)

y_prec = te.label.isin(["rain", "snow"]).values
fig, ax = plt.subplots(figsize=(4.5, 4))
for bal, lab in [(True, "balanced"), (False, "без весов")]:
    mm, _, _ = fit_eval(FEATS_ONBOARD, "hgb", balanced=bal)
    pp = p_precip(mm, te, FEATS_ONBOARD)
    x, y, _ = reliability_curve(y_prec, pp)
    ax.plot(x, y, "o-", label=f"{lab}, ECE={ece(y_prec, pp):.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("предсказанная вероятность"); ax.set_ylabel("доля осадков")
ax.legend(); ax.set_title("Reliability: P(осадки)"); plt.show()
m_cal = mm   # модель без весов используем для вероятностей на выходе

## 7. Выход модуля: вероятности → гистерезис → алерт
Покажем на 10-дневном фрагменте теста, как вероятность осадков превращается в устойчивое состояние (без дрожания на границе).

In [ ]:
st0 = STATIONS[0]
seg = te[te.station == st0].iloc[24 * 100: 24 * 110]
pp = p_precip(m_cal, seg, FEATS_ONBOARD)
state = hysteresis(pp, on=0.6, off=0.3)
fig, axs = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axs[0].plot(seg.index, pp, label="P(осадки)"); axs[0].fill_between(seg.index, 0, seg.label.isin(["rain", "snow"]) * 1.0, alpha=.3, color="gray", step="mid", label="истина")
axs[0].axhline(.6, c="r", ls=":"); axs[0].axhline(.3, c="g", ls=":"); axs[0].legend(ncol=2)
axs[1].fill_between(seg.index, 0, state * 1.0, step="mid", color="C1"); axs[1].set_ylabel("состояние")
plt.tight_layout(); plt.show()

def condition_message(row, probs, uav_id="UAV-001"):
    """Формат под БВС-3 (alert_type уточнить: в ТЗ нет weather)."""
    top = max(probs, key=probs.get)
    return {"alert_id": f"wx-{row.name:%Y%m%d%H}", "uav_id": uav_id, "alert_type": None,
            "description": f"WX_{top.upper()}", "sensor_source": "env_sensors",
            "confidence": round(probs[top], 2), "timestamp": int(row.name.timestamp() * 1000),
            "probs": {k: round(v, 2) for k, v in probs.items()}}

r = seg.iloc[int(np.argmax(pp))]
P1 = dict(zip(m_cal.classes_, m_cal.predict_proba(seg.loc[[r.name], FEATS_ONBOARD])[0]))
condition_message(r, P1)

## 8. Выводы и ограничения
*(Заполните после запуска на реальных данных IEM; числа зависят от источника, указанного в разделе 1.)*

- Сравните `rule` и `logreg/hgb`: прирост показывает, сколько даёт обучение сверх физики Tw.
- Сравните наборы `raw → onboard → +vis`: вклад производных признаков и внешней видимости.
- Редкие классы (`hail_ts`, `aerosol`) в Германии малочисленны: смотрите recall по ним отдельно, macro-F1 на них нестабилен.
- Метки METAR привязаны к станции; переносимость на дрон на высоте 100-1000 м не проверена.
- Для **града** результат полезен только как предполётный риск (T, RH, ветер, порывы): в полёте ничего не меняет (ноутбук 9 плана).